# 3. Figures

**Paper:** AUROC1 (Family / Superfamily / Fold) and AA↔3Di translation accuracy bars.

This notebook only **reads** metrics. It does not search or score sequences.

| | Path |
|--|------|
| Input | `work/metrics/{method}_easy_{fam,sup,fol}.tsv` from `2a_remote_homology.ipynb` |
| | `work/metrics/translation/translation_summary.csv` from `2b_translation_accuracy.ipynb` (optional) |
| Output | `work/figures/auroc1_easy.png` / `.pdf` |
| | `work/metrics/auc_easy.csv` (refreshed from the curves) |
| | `work/figures/translation_accuracy.png` / `.pdf` (if translation summary exists) |

**Legend:** solid = **hitlist** (Foldseek / predicted 3Di); dashed = **catalog** (MMseqs2). The two protocols are not the same AUC.

**Node:** login is enough. Seconds.


## Environment


In [ ]:
NOTEBOOK_NAME = "3_figures.ipynb"

import os
import platform
import subprocess
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
if not (cwd / NOTEBOOK_NAME).is_file():
    raise SystemExit(
        f"Start this notebook from the project root (cwd must contain {NOTEBOOK_NAME}). "
        f"Current cwd: {cwd}"
    )

CONDA_ENV = "ESM3_3Di_5090"
print("notebook:", NOTEBOOK_NAME)
print("cwd:", cwd)
print("python:", sys.executable)
print("version:", sys.version.split()[0])
print("platform:", platform.platform())
print("CONDA_DEFAULT_ENV:", os.environ.get("CONDA_DEFAULT_ENV", "(unset)"))

if CONDA_ENV not in sys.executable:
    expected = Path.home() / ".conda" / "envs" / CONDA_ENV / "bin" / "python"
    raise SystemExit(
        f"Kernel is not {CONDA_ENV} (current: {sys.executable}). "
        f"Select kernel {CONDA_ENV} and Restart. Do not pip into miniforge3 python3.12. "
        f"Expected: {expected}"
    )


def _bin_version(name: str) -> str:
    path = cwd / "bin" / name
    if not path.is_file():
        return "(not installed yet; run 0_prepare_scope40.ipynb)"
    try:
        proc = subprocess.run([str(path), "version"], capture_output=True, text=True, check=False)
        lines = (proc.stdout or proc.stderr or "").strip().splitlines()
        return lines[0] if lines else "(unknown)"
    except OSError as exc:
        return f"(failed: {exc})"


print("foldseek:", _bin_version("foldseek"))
print("mmseqs:", _bin_version("mmseqs"))


## Configuration

本格在五本 notebook 中**字节级相同**。改方法表、搜索参数或 URL 时：只改 `0_prepare_scope40.ipynb` 这一格，再整格复制到另外四本。发布前可用 checksum 核对五本是否一致。


In [ ]:
# =============================================================================
# Configuration — copy this entire cell into all five notebooks.
# Change methods or search parameters here in 0_prepare_scope40.ipynb, then
# paste the same cell into 1_build / 2a / 2b / 3_figures.
# =============================================================================
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
HOME = ROOT.parent

CONDA_ENV = "ESM3_3Di_5090"
FOLDSEEK_VERSION = "10-941cd33"
MMSEQS_VERSION = "18-8cc5c"

TEMP = ROOT / "tmp"
WORK_DIR = ROOT / "work"
BIN_DIR = ROOT / "bin"
WORK_TMP_DIR = WORK_DIR / "tmp"

GT_FASTA_DIR = WORK_DIR / "GT_fasta"
AA_FASTA = GT_FASTA_DIR / "DB_aa.fasta"
GT_DI_FASTA = GT_FASTA_DIR / "DB_di.fasta"

AA2DI_FASTA_DIR = WORK_DIR / "aa2di_fasta"
DI2AA_FASTA_DIR = WORK_DIR / "di2aa_fasta"

DBS_DIR = WORK_DIR / "DB"
FOLDSEEK_GT_DIR = DBS_DIR / "foldseek_DB"
MMSEQS_GT_DIR = DBS_DIR / "mmseqs_DB"

LABEL_DIR = WORK_DIR / "labels"
SCOP_LOOKUP = LABEL_DIR / "scop_lookup.tsv"
LEGACY_LABEL_DIR = WORK_DIR / "lable"

ALN_DIR = WORK_DIR / "aln"
METRICS_DIR = WORK_DIR / "metrics"
FIGURES_DIR = WORK_DIR / "figures"
TRANSLATION_METRICS_DIR = METRICS_DIR / "translation"
WORK_BUNDLE = WORK_DIR / "scope40_work_bundle.tar.gz"

FOLDSEEK_BIN = BIN_DIR / "foldseek"
MMSEQS_BIN = BIN_DIR / "mmseqs"

FOLDSEEK_URL = (
    "https://github.com/steineggerlab/foldseek/releases/download/"
    f"{FOLDSEEK_VERSION}/foldseek-linux-avx2.tar.gz"
)
MMSEQS_URL = (
    "https://github.com/soedinglab/MMseqs2/releases/download/"
    f"{MMSEQS_VERSION}/mmseqs-linux-avx2.tar.gz"
)
FOLDSEEK_TMP_DIR = TEMP / "foldseek"
MMSEQS_TMP_DIR = TEMP / "mmseqs"
FOLDSEEK_TARBALL = TEMP / "foldseek-linux-avx2.tar.gz"
MMSEQS_TARBALL = TEMP / "mmseqs-linux-avx2.tar.gz"

SCOP_CLA_NAME = "dir.cla.scope.2.08-stable.txt"
SCOP_DES_NAME = "dir.des.scope.2.08-stable.txt"
SOURCE_ARCHIVE_NAME = "pdbstyle-sel-gs-bib-40-2.08.tgz"
SCOP_CLA_FALLBACK = HOME / "SCOPE" / SCOP_CLA_NAME
HF_BASE = "https://huggingface.co/datasets/caijihuize/scope40_pdbstyle/resolve/main"

# Foldseek / predicted 3Di: aligned with new_scope40 easy-search
EASY_SEARCH_PARAMS = {
    "sensitivity": 9.5,
    "max_seqs": 2000,
    "evalue": 10.0,
    "threads": 64,
}
# MMseqs2: aligned with foldseek-analysis/scopbenchmark/scripts/runMMseqs.sh
MMSEQS_SEARCH_PARAMS = {
    "sensitivity": 7.5,
    "max_seqs": 2000,
    "evalue": 10000,
    "threads": 64,
    "add_backtrace": True,
}
PREPARE_THREADS = 16
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

# Homology search methods. protocol must not be mixed as one AUC.
METHODS: list[dict] = [
    {"name": "Foldseek (AA+3Di)", "key": "foldseek", "engine": "foldseek", "aa2di": None, "protocol": "hitlist"},
    {"name": "MMseqs2", "key": "mmseqs", "engine": "mmseqs", "aa2di": None, "protocol": "catalog"},
    {"name": "ESM3-3Di", "key": "ESM3", "engine": "foldseek", "aa2di": "DB_ESM3_aa2di.fasta", "protocol": "hitlist"},
    {"name": "ESM3-LoRA", "key": "ESM3_LoRA", "engine": "foldseek", "aa2di": "DB_ESM3_LoRA_aa2di.fasta", "protocol": "hitlist"},
    {"name": "ProstT5 (translate)", "key": "ProstT5", "engine": "foldseek", "aa2di": "DB_ProstT5_translate_aa2di.fasta", "protocol": "hitlist"},
    {"name": "SaProt", "key": "SaProt", "engine": "foldseek", "aa2di": "DB_SaProt_aa2di.fasta", "protocol": "hitlist"},
]
# Translation accuracy (bidirectional). Homology search uses aa2di only.
TRANSLATION_METHODS: list[dict] = [
    {"name": "ESM3-3Di", "key": "ESM3", "aa2di": "DB_ESM3_aa2di.fasta", "di2aa": "DB_ESM3_di2aa.fasta"},
    {"name": "ESM3-LoRA", "key": "ESM3_LoRA", "aa2di": "DB_ESM3_LoRA_aa2di.fasta", "di2aa": "DB_ESM3_LoRA_di2aa.fasta"},
    {"name": "ProstT5 (translate)", "key": "ProstT5", "aa2di": "DB_ProstT5_translate_aa2di.fasta", "di2aa": "DB_ProstT5_translate_di2aa.fasta"},
    {"name": "SaProt", "key": "SaProt", "aa2di": "DB_SaProt_aa2di.fasta", "di2aa": "DB_SaProt_di2aa.fasta"},
]
PALETTE = {
    "Foldseek (AA+3Di)": "#2b5c8f",
    "MMseqs2": "#666666",
    "ESM3-3Di": "#d95f02",
    "ESM3-LoRA": "#1b9e77",
    "ProstT5 (translate)": "#7570b3",
    "SaProt": "#e7298a",
}


def run_cmd(argv: list[str]) -> None:
    """Print then run an external command. Logs are supplementary material."""
    print("[CMD]", " ".join(str(x) for x in argv), flush=True)
    subprocess.run([str(x) for x in argv], check=True)


def require_file(path: Path, hint: str) -> Path:
    """Fail with a pointer to the upstream notebook if a required file is missing."""
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}\n{hint}")
    return path


def meta_path(output: Path) -> Path:
    return output.with_name(output.name + ".meta.json")


def skip_if_exists(output: Path, payload: dict | None = None, skip_existing: bool = True) -> bool:
    """Skip when output exists. If payload is given, require a matching .meta.json.

    MMseqs TSV without a fingerprint is treated as stale (catalog protocol change).
    Other outputs without a fingerprint are kept; set SKIP_EXISTING=False to force.
    """
    if not skip_existing:
        return False
    if not output.is_file() or output.stat().st_size == 0:
        return False
    if payload is None:
        return True
    meta = meta_path(output)
    if not meta.is_file():
        if payload.get("engine") == "mmseqs":
            print(f"[rerun] {output.name}: no fingerprint; MMseqs catalog params need a fresh search")
            return False
        print(f"[warn] {output.name}: no fingerprint; keeping existing file (set SKIP_EXISTING=False to re-run)")
        return True
    try:
        stored = json.loads(meta.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        return False
    if stored != payload:
        print(f"[rerun] {output.name}: Configuration changed")
        return False
    return True


def write_meta(output: Path, payload: dict) -> None:
    meta_path(output).write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def search_params_payload(engine: str, threads: int | None = None) -> dict:
    params = dict(MMSEQS_SEARCH_PARAMS if engine == "mmseqs" else EASY_SEARCH_PARAMS)
    params["engine"] = engine
    if threads is not None:
        params["threads"] = int(threads)
    return params


def method_by_key(method_key: str) -> dict:
    for row in METHODS:
        if row["key"] == method_key:
            return row
    raise KeyError(f"Unknown method_key: {method_key}")


def predicted_methods() -> list[dict]:
    return [row for row in METHODS if row["aa2di"] is not None]


def db_prefix(method_key: str) -> Path:
    return DBS_DIR / f"{method_key}_DB" / "DB"


def aln_tsv(method_key: str) -> Path:
    return ALN_DIR / f"{method_key}_easy.tsv"


def aln_tmp_dir(method_key: str) -> Path:
    return WORK_TMP_DIR / f"easy_{method_key}"


def metric_prefix(method_key: str) -> Path:
    return METRICS_DIR / f"{method_key}_easy"


def translation_per_seq_path(task: str, method_key: str) -> Path:
    return TRANSLATION_METRICS_DIR / f"{task}_{method_key}_per_seq.tsv"


def translation_summary_path(task: str) -> Path:
    return TRANSLATION_METRICS_DIR / f"{task}_summary.csv"


def scop_cla_path() -> Path:
    for path in (TEMP / SCOP_CLA_NAME, SCOP_CLA_FALLBACK):
        if path.is_file():
            return path
    return TEMP / SCOP_CLA_NAME


def work_ready() -> bool:
    return (
        (FOLDSEEK_GT_DIR / "DB").is_file()
        and (MMSEQS_GT_DIR / "DB").is_file()
        and AA_FASTA.is_file()
        and GT_DI_FASTA.is_file()
        and SCOP_LOOKUP.is_file()
    )


def ensure_work_dirs() -> None:
    for directory in (
        TEMP, BIN_DIR, GT_FASTA_DIR, AA2DI_FASTA_DIR, DI2AA_FASTA_DIR, DBS_DIR,
        LABEL_DIR, ALN_DIR, METRICS_DIR, TRANSLATION_METRICS_DIR, FIGURES_DIR, WORK_TMP_DIR,
    ):
        directory.mkdir(parents=True, exist_ok=True)
    legacy = LEGACY_LABEL_DIR / "scop_lookup.tsv"
    if not SCOP_LOOKUP.is_file() and legacy.is_file():
        shutil.copy2(legacy, SCOP_LOOKUP)
        print(f"[ok] migrated {legacy} -> {SCOP_LOOKUP}")


def cleanup_tmp(*, also_work_tmp: bool = True) -> None:
    """Remove tmp/ and work/tmp/ only. Keep work/ products and bin/."""
    targets = [TEMP]
    if also_work_tmp:
        targets.append(WORK_TMP_DIR)
    for path in targets:
        if path.exists():
            shutil.rmtree(path)
            print(f"[ok] cleaned {path}")
        else:
            print(f"[skip] {path} (absent)")


print("ROOT:", ROOT)
print("conda env:", CONDA_ENV)
print("Foldseek:", FOLDSEEK_VERSION, "MMseqs:", MMSEQS_VERSION)
print("METHODS:", [(m["key"], m["engine"], m["protocol"]) for m in METHODS])


## Run flags


In [ ]:
SAVE_PDF = True

ensure_work_dirs()
print("metrics:", METRICS_DIR)
print("figures:", FIGURES_DIR)
print("SAVE_PDF:", SAVE_PDF)


## Helpers


In [ ]:
import pandas as pd

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError as exc:
    raise SystemExit(
        "matplotlib is missing. Select kernel ESM3_3Di_5090; "
        "do not pip into cluster miniforge3 python3.12."
    ) from exc

try:
    from IPython.display import Image, display
except ImportError:
    def display(obj):
        print(obj)
    Image = None


def save_fig(fig, png_path: Path) -> None:
    png_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(png_path, dpi=300, facecolor="white", bbox_inches="tight")
    print(f"[ok] {png_path}")
    if SAVE_PDF:
        pdf_path = png_path.with_suffix(".pdf")
        fig.savefig(pdf_path, facecolor="white", bbox_inches="tight")
        print(f"[ok] {pdf_path}")


print("helpers ready")


## Step 1 — AUROC1 (remote homology)

Each curve is the query sensitivity ranked descending. AUC = mean query sensitivity. MMseqs2 is drawn dashed and labelled `[catalog]`.


In [ ]:
def roc_plot(ax, file: Path, tool: str, color: str | None = None, linestyle: str = "-") -> float:
    data = []
    with file.open() as handle:
        for line in handle:
            parts = line.strip().split()
            if len(parts) >= 4:
                data.append(parts)
    if not data:
        return 0.0
    data.sort(key=lambda x: float(x[3]), reverse=True)
    x = [(i + 1) / len(data) for i in range(len(data))]
    y = [float(row[3]) for row in data]
    auc_val = sum(y) / len(y)
    ax.plot(x, y, label=f"{tool} AUC={auc_val:.3f}", color=color, linewidth=1.4, linestyle=linestyle)
    return auc_val


def plot_auroc1_easy(
    output_png: Path | None = None,
    auc_csv: Path | None = None,
) -> tuple[Path, Path]:
    output_png = Path(output_png or (FIGURES_DIR / "auroc1_easy.png"))
    auc_csv = Path(auc_csv or (METRICS_DIR / "auc_easy.csv"))
    level_files = {"Family": "fam", "Superfamily": "sup", "Fold": "fol"}
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))
    auc_rows: list[dict] = []

    for ax, (title, level_key) in zip(axs, level_files.items()):
        ax.set_title(title, fontsize=16)
        row: dict[str, float | str] = {"search_mode": "easy", "level": title}
        for method in METHODS:
            path = Path(str(metric_prefix(method["key"])) + f"_{level_key}.tsv")
            if not path.is_file():
                print(f"[missing] {method['name']} {title}: {path}")
                continue
            linestyle = "--" if method["protocol"] == "catalog" else "-"
            legend = f"{method['name']} [{method['protocol']}]"
            auc_val = roc_plot(ax, path, legend, color=PALETTE.get(method["name"]), linestyle=linestyle)
            row[method["name"]] = auc_val
            print(f"[ok] {method['name']} {title} [{method['protocol']}]: AUC={auc_val:.4f}")
        auc_rows.append(row)
        ax.set_xlim(-0.01, 1.01)
        ax.set_ylim(-0.01, 1.01)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        legend_loc = "upper right" if title == "Fold" else "lower left"
        ax.legend(fontsize=7, loc=legend_loc)

    axs[0].set_ylabel("Fraction of TPs up to first FP", fontsize=12)
    axs[1].set_xlabel("Fraction of queries", fontsize=12)
    fig.suptitle(
        "Solid = hitlist (Foldseek / predicted 3Di). Dashed = catalog (MMseqs2). "
        "The two protocols are not the same metric.",
        fontsize=10,
        y=1.02,
    )
    save_fig(fig, output_png)
    plt.close(fig)

    df = pd.DataFrame(auc_rows)
    df.to_csv(auc_csv, index=False)
    print(f"[ok] AUC CSV: {auc_csv}")
    return output_png, auc_csv


png_path, csv_path = plot_auroc1_easy()
auc_df = pd.read_csv(csv_path)
display(auc_df)
if Image is not None:
    display(Image(filename=str(png_path)))


## Step 2 — Translation accuracy bars

Reads `translation_summary.csv` from 2b. Skipped if that file is absent.


In [ ]:
def plot_translation_accuracy(df: pd.DataFrame, output_png: Path | None = None) -> Path | None:
    if df.empty:
        print("[skip] no translation data (run 2b_translation_accuracy.ipynb)")
        return None
    output_png = Path(output_png or (FIGURES_DIR / "translation_accuracy.png"))
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, task, title in zip(
        axes,
        ("aa2di", "di2aa"),
        ("AA→3Di vs GT 3Di", "3Di→AA vs GT AA"),
        strict=True,
    ):
        sub = df[df["task"] == task].sort_values("micro_acc", ascending=False)
        if sub.empty:
            ax.set_title(f"{title} (no data)")
            continue
        x = range(len(sub))
        colors = [PALETTE.get(lbl, "#888888") for lbl in sub["label"]]
        ax.bar(x, sub["micro_acc"], color=colors, alpha=0.85, label="micro")
        ax.plot(x, sub["macro_acc"], "ko-", markersize=6, label="macro")
        ax.set_xticks(list(x))
        ax.set_xticklabels(sub["label"], rotation=25, ha="right")
        ax.set_ylim(0, 1.02)
        ax.set_ylabel("Accuracy")
        ax.set_title(title)
        ax.legend(loc="lower right")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    fig.tight_layout()
    save_fig(fig, output_png)
    plt.close(fig)
    return output_png


trans_csv = TRANSLATION_METRICS_DIR / "translation_summary.csv"
if trans_csv.is_file():
    trans_df = pd.read_csv(trans_csv)
else:
    trans_df = pd.DataFrame()
    print(f"[skip] missing {trans_csv}")

trans_png = plot_translation_accuracy(trans_df)
if trans_png is not None:
    display(trans_df)
    if Image is not None:
        display(Image(filename=str(trans_png)))


## Verify


In [ ]:
print("AUROC1 PNG:", (FIGURES_DIR / "auroc1_easy.png").is_file())
print("AUC CSV:", (METRICS_DIR / "auc_easy.csv").is_file())
print("translation PNG:", (FIGURES_DIR / "translation_accuracy.png").is_file())
require_file(FIGURES_DIR / "auroc1_easy.png", hint="Run Step 1; need 2a metrics.")
require_file(METRICS_DIR / "auc_easy.csv", hint="Run Step 1.")
print("[ok] figures written under work/figures/")
print("Protocols: hitlist (solid) vs catalog (dashed) — do not mix as one table without a note.")


## Cleanup


In [ ]:
cleanup_tmp(also_work_tmp=True)
print("Cleanup done. Products remain under work/ and bin/.")
